# Operators Built-in Validation


## 1. Environment and paths


In [ ]:
import os
import shutil
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import load_dotenv

from image_gallery.cleaning import BasicCleaner
from image_gallery.dataset import Dataset
from image_gallery.storage import MinioStorage


repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

LOCAL_RAW_DATASET_PATH = repo_root / "notebooks" / ".importers_test_library" / "outputs" / "raw.parquet"
MINIO_RAW_DATASET_PATH = repo_root / "notebooks" / ".importers_test_library" / "minio_outputs" / "raw.parquet"
LIBRARY_ROOT = repo_root / "notebooks" / ".operators_test_library"
OUTPUT_DIR = LIBRARY_ROOT / "outputs"
MINIO_CACHE_DIR = LIBRARY_ROOT / "minio_cache"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MINIO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RAW_DATASET_PATH, MINIO_RAW_DATASET_PATH, OUTPUT_DIR, MINIO_CACHE_DIR


## 2. Load local raw Dataset


In [ ]:
def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def require_columns(frame: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise AssertionError(f"{label} missing required columns: {missing}")


require_file(LOCAL_RAW_DATASET_PATH, "local raw dataset")
local_dataset = Dataset.from_path(str(LOCAL_RAW_DATASET_PATH))
local_raw_frame = local_dataset.to_frame()
require_columns(local_raw_frame, ["image_id", "image_uri"], "local raw dataset")

print("local rows:", len(local_raw_frame))
print("local columns:", local_raw_frame.columns.tolist())
local_raw_frame.head()


In [ ]:
require_file(MINIO_RAW_DATASET_PATH, "MinIO raw dataset")
minio_raw_dataset = Dataset.from_path(str(MINIO_RAW_DATASET_PATH))
minio_raw_frame = minio_raw_dataset.to_frame()
require_columns(minio_raw_frame, ["image_id", "image_uri"], "MinIO raw dataset")

print("minio rows:", len(minio_raw_frame))
print("minio columns:", minio_raw_frame.columns.tolist())
minio_raw_frame[["image_id", "image_uri"]].head()


## 3. Load and materialize MinIO raw Dataset


In [ ]:
load_dotenv(repo_root / ".env")


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


raw_endpoint = require_env("IMAGE_GALLERY_MINIO_ENDPOINT")
access_key = require_env("IMAGE_GALLERY_MINIO_ACCESS_KEY")
secret_key = require_env("IMAGE_GALLERY_MINIO_SECRET_KEY")
bucket = require_env("IMAGE_GALLERY_MINIO_BUCKET")

parsed = urlparse(raw_endpoint)
endpoint = parsed.netloc or raw_endpoint
secure = parsed.scheme == "https"

minio_storage = MinioStorage(storage_name="operator_validation_minio").connect(
    endpoint=endpoint,
    access_key=access_key,
    secret_key=secret_key,
    bucket=bucket,
    secure=secure,
)

print("connected bucket:", minio_storage.bucket)


In [ ]:
def parse_s3_uri(image_uri: str, expected_bucket: str) -> str:
    parsed_uri = urlparse(image_uri)
    if parsed_uri.scheme != "s3":
        raise ValueError(f"expected s3 URI, got: {image_uri}")
    if parsed_uri.netloc != expected_bucket:
        raise ValueError(f"s3 URI bucket {parsed_uri.netloc!r} does not match env bucket {expected_bucket!r}")
    return parsed_uri.path.lstrip("/")


def materialize_minio_dataset(frame: pd.DataFrame, storage: MinioStorage, cache_dir: Path, bucket_name: str) -> Dataset:
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    materialized = frame.copy()
    original_uris: list[str] = []
    local_uris: list[str] = []
    for row in frame.itertuples(index=False):
        original_uri = str(row.image_uri)
        object_path = parse_s3_uri(original_uri, bucket_name)
        extension = Path(object_path).suffix or ".img"
        local_path = cache_dir / f"{row.image_id}{extension}"
        try:
            data = storage.read_bytes(object_path)
        except Exception as exc:
            raise RuntimeError(f"failed to read MinIO object {object_path}") from exc
        local_path.write_bytes(data)
        original_uris.append(original_uri)
        local_uris.append(str(local_path))

    materialized["minio_image_uri"] = original_uris
    materialized["image_uri"] = local_uris
    output_path = cache_dir.parent / "minio_materialized_raw.parquet"
    return Dataset.write(materialized, str(output_path))


minio_materialized_dataset = materialize_minio_dataset(
    minio_raw_frame,
    minio_storage,
    MINIO_CACHE_DIR,
    bucket,
)
minio_materialized_frame = minio_materialized_dataset.to_frame()
minio_materialized_frame[["image_id", "image_uri", "minio_image_uri"]].head()


## 4. Run BasicCleaner


In [ ]:
OPERATOR_CONFIGS = [
    {"format.decode_check": {}},
    {"size.dimension_check": {"min_width": 1, "min_height": 1, "action": "review"}},
    {"quality.blur_check": {"threshold": 0.0, "action": "review"}},
    {"quality.brightness_check": {"min_threshold": 0.0, "max_threshold": 255.0, "action": "review"}},
    {"quality.contrast_check": {"threshold": 0.0, "action": "review"}},
    {"duplicate.exact_duplicate_check": {"action": "review"}},
]


def run_cleaner(label: str, dataset: Dataset) -> dict[str, object]:
    run_output_dir = OUTPUT_DIR / label
    if run_output_dir.exists():
        shutil.rmtree(run_output_dir)
    cleaner = BasicCleaner(OPERATOR_CONFIGS)
    cleaner.run(dataset, output_dir=run_output_dir)
    context = cleaner._context
    if context is None:
        raise AssertionError(f"{label} cleaner context should exist")
    paths = context.paths
    return {
        "label": label,
        "cleaner": cleaner,
        "paths": paths,
        "parameter_table": pd.read_parquet(paths.parameter_table_path),
        "evaluation_table": pd.read_parquet(paths.evaluation_table_path),
        "preview": cleaner.preview(),
        "state": cleaner.state(),
    }


local_result = run_cleaner("local", local_dataset)
minio_result = run_cleaner("minio", minio_materialized_dataset)
local_result["state"], minio_result["state"]


In [ ]:
def validate_stage3_core(result: dict[str, object]) -> None:
    label = str(result["label"])
    cleaner = result["cleaner"]
    paths = result["paths"]
    preview = result["preview"]
    state = result["state"]

    for path in [
        paths.parameter_table_path,
        paths.evaluation_table_path,
        paths.operator_outputs_path,
        paths.state_path,
    ]:
        if not path.exists():
            raise AssertionError(f"{label} missing output: {path}")

    expected_operators = [next(iter(item.keys())) for item in OPERATOR_CONFIGS]
    if state["operator_name"].tolist() != expected_operators:
        raise AssertionError(f"{label} state operator order mismatch")
    if not {"clean_count", "review_count", "dropped_count", "restricted_count"}.issubset(set(vars(preview))):
        raise AssertionError(f"{label} preview missing count attributes")

    for operator_name in expected_operators:
        result_frame = cleaner.result(operator_name)
        if not {"image_id", "image_uri"}.issubset(result_frame.columns):
            raise AssertionError(f"{label} result missing identity columns for {operator_name}")


validate_stage3_core(local_result)
validate_stage3_core(minio_result)
print("stage3 core validation ok")


## 5. Validate stage 3 cleaning core


## 6. Validate stage 4 built-in operators


In [ ]:
PARAMETER_COLUMNS = [
    "width",
    "height",
    "decode_ok",
    "decode_error",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "content_hash",
    "phash",
    "exact_duplicate_group_id",
]

EVALUATION_COLUMNS = [
    "decode_action",
    "decode_reason",
    "dimension_action",
    "dimension_reason",
    "blur_action",
    "blur_reason",
    "brightness_action",
    "brightness_reason",
    "contrast_action",
    "contrast_reason",
    "exact_duplicate_action",
    "exact_duplicate_reason",
    "final_action",
    "final_reason",
    "triggered_operator_names",
]


def validate_stage4_columns(result: dict[str, object]) -> None:
    label = str(result["label"])
    parameter_table = result["parameter_table"]
    evaluation_table = result["evaluation_table"]
    require_columns(parameter_table, PARAMETER_COLUMNS, f"{label} parameter_table")
    require_columns(evaluation_table, EVALUATION_COLUMNS, f"{label} evaluation_table")


validate_stage4_columns(local_result)
validate_stage4_columns(minio_result)
local_result["parameter_table"].head(), minio_result["parameter_table"].head()


## 7. Validate config/rerun


In [ ]:
local_cleaner = local_result["cleaner"]
local_cleaner.config([{"quality.blur_check": {"threshold": 999999.0, "action": "review"}}])
stale_state = local_cleaner.state()
if stale_state.loc[stale_state["operator_name"] == "quality.blur_check", "status"].item() != "stale":
    raise AssertionError("quality.blur_check should be stale after config()")

local_cleaner.rerun([{"quality.blur_check": {"threshold": 999999.0, "action": "review"}}])
completed_state = local_cleaner.state()
if completed_state.loc[completed_state["operator_name"] == "quality.blur_check", "status"].item() != "completed":
    raise AssertionError("quality.blur_check should be completed after rerun()")

local_result["parameter_table"] = pd.read_parquet(local_result["paths"].parameter_table_path)
local_result["evaluation_table"] = pd.read_parquet(local_result["paths"].evaluation_table_path)
local_result["preview"] = local_cleaner.preview()
completed_state


## 8. Export views


In [ ]:
def export_and_validate_counts(result: dict[str, object]) -> dict[str, int]:
    label = str(result["label"])
    cleaner = result["cleaner"]
    export_dir = OUTPUT_DIR / label / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    datasets = {
        "full": cleaner.export("full", str(export_dir / "full.parquet")),
        "clean": cleaner.export("clean", str(export_dir / "clean.parquet")),
        "review": cleaner.export("review", str(export_dir / "review.parquet")),
        "dropped": cleaner.export("dropped", str(export_dir / "dropped.parquet")),
        "parameters": cleaner.export("parameters", str(export_dir / "parameters.parquet")),
        "evaluations": cleaner.export("evaluations", str(export_dir / "evaluations.parquet")),
    }
    counts = {name: dataset.count() for name, dataset in datasets.items()}
    restricted_count = result["preview"].restricted_count
    if counts["full"] != counts["clean"] + counts["review"] + counts["dropped"] + restricted_count:
        raise AssertionError(f"{label} export counts do not add up: {counts}, restricted={restricted_count}")
    if counts["parameters"] != counts["full"]:
        raise AssertionError(f"{label} parameters count should equal full count")
    if counts["evaluations"] != counts["full"]:
        raise AssertionError(f"{label} evaluations count should equal full count")
    return counts


local_counts = export_and_validate_counts(local_result)
minio_counts = export_and_validate_counts(minio_result)
local_counts, minio_counts


In [ ]:
print("PASS: stage3 and stage4 operator validation completed")


## 9. PASS
